# Deployment

Let's deploy the Foundation AI Foundation-Sec-8B-Reasoning model onto Amazon SageMaker AI. <br>
You can use the model deployed by this notebook for inference. Refer to [the inference notebook](https://github.com/RobustIntelligence/foundation-ai-cookbook/blob/main/3_adoptions/deployment/sagemaker/foundation_sec_8b_reasoning/inference.ipynb) for sample code.

This is a reasoning model that produces chain-of-thought tokens during inference. It requires:
- The **vLLM** container image (with the `minimax_m2` reasoning parser)
- A **g6e** family instance or better (e.g., `ml.g6e.2xlarge`) to handle the higher token throughput

As a prerequisite, please launch JupyterLab on SageMaker in your AWS environment. For more details, visit: 
https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-jl.html

In [ ]:
import sagemaker
import boto3

In [ ]:
MODEL_NAME = 'fdtn-ai/Foundation-Sec-8B-Reasoning'
INSTANCE_TYPE = 'ml.g6e.2xlarge'

In [ ]:
try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client('iam')
    role = iam.get_role(RoleName='sagemaker_execution_role')['Role']['Arn']

We use the AWS-released vLLM container image. The reasoning model requires the `minimax_m2` reasoning parser,
which we configure via the `SM_VLLM_REASONING_PARSER` environment variable.

In [ ]:
# Replace us-west-2 with your AWS region in the ECR URI below
container_uri = "763104351884.dkr.ecr.us-west-2.amazonaws.com/vllm:0.17.0-gpu-py312-cu129-ubuntu22.04-sagemaker"

In [ ]:
endpoint_name = sagemaker.utils.name_from_base("Foundation-Sec-8B-Reasoning")
print("Deploying to endpoint:", endpoint_name)

In [ ]:
env_vars = {
    "SM_VLLM_MODEL": MODEL_NAME,
    "SM_VLLM_REASONING_PARSER": "minimax_m2",
    "SM_VLLM_MAX_MODEL_LEN": "32768",
    "SM_VLLM_TRUST_REMOTE_CODE": "true",
}

model = sagemaker.Model(
    name=endpoint_name,
    image_uri=container_uri,
    role=role,
    env=env_vars,
)

model.deploy(
    instance_type=INSTANCE_TYPE,
    initial_instance_count=1,
    endpoint_name=endpoint_name,
)

The predictor's endpoint will be used for [inference](https://github.com/RobustIntelligence/foundation-ai-cookbook/blob/main/3_adoptions/deployment/sagemaker/foundation_sec_8b_reasoning/inference.ipynb). You can also get the endpoint from SageMaker Studio's console.

In [ ]:
# Uncomment to delete the endpoint and avoid ongoing costs
# sagemaker.Session().delete_endpoint(endpoint_name)